In [1]:
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import xgboost as xgb
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.datasets import make_regression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_curve, auc
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from string import ascii_uppercase
from geopy.distance import geodesic

from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

c:\Users\rah10\miniconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
X = np.load('../pollenGeolocation_saved/X.npy')
y = np.load('../pollenGeolocation_saved/y.npy')

In [3]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1298)

# Create scaler for X and y
sc_X = StandardScaler()
sc_y = StandardScaler()

# Scale training data
X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)  

# Scale test data using training parameters
X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)  

In [4]:
def construct_model():
    # the list of classifiers to use
    regression_models = [
        linear_model.MultiTaskLasso(),
        MultiOutputRegressor(SVR())
    ]

    linear_model_parameters = {}

    lasso_parameters = {
        'alpha': np.arange(0.00, 1.0, 0.01)
    }

    svr_parameters = {
        'estimator__epsilon': np.linspace(0.001, 1, 10),
        'estimator__kernel': ['linear'],
        'estimator__C': np.linspace(0.01, 100, 10)
    }
    parameters = [
        lasso_parameters,
        svr_parameters
    ]
    data['estimators'] = []
    # iterate through each classifier and use GridSearchCV
    for i, regressor in enumerate(regression_models):
        clf = GridSearchCV(regressor,              # model
                  param_grid = parameters[i], # hyperparameters
                  scoring='neg_mean_absolute_error',         # metric for scoring
                  cv=10,
                  n_jobs=-1, error_score='raise', verbose=3)
        clf.fit(X_train, y_train)
        # add the clf to the estimators list
        data['estimators'].append((regressor.__class__.__name__, clf))

In [5]:
data = {}
construct_model()

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Fitting 10 folds for each of 100 candidates, totalling 1000 fits


In [6]:
for estimator in data['estimators']:
  print('Best Score for %s: %s' % (estimator[0], estimator[1].best_score_))
  print('Best Hyperparameters for %s: %s' % (estimator[0], estimator[1].best_params_))
  print("\n")

Best Score for MultiTaskLasso: -0.08677049729288025
Best Hyperparameters for MultiTaskLasso: {'alpha': 0.01}


Best Score for MultiOutputRegressor: -0.07225225150444294
Best Hyperparameters for MultiOutputRegressor: {'estimator__C': 0.01, 'estimator__epsilon': 0.001, 'estimator__kernel': 'linear'}




In [ ]:
np.arange(0.001, 1.0, 1)